In [46]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
import time

woc = WocMapsRemote(base_url="https://worldofcode.org/api/")

projects = [
    'gabinsane/multimodal-vae-comparison',
    'MansMeg/posteriordb',
    'markmfredrickson/optmatch',
    'statistikat/VIM',
    'FairRootGroup/FairRoot',
    'arfc/pride',
    'nexpy/nexpy',
    'tompollard/tableone',
    'geospace-code/pymap3d',
    'GeoKnow/LinkedGeoData'
]

list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head()



,sha1,project
0,034f34e1158ffb69f5ad4ceefeff4991e312abc8,gabinsane_multimodal-vae-comparison
1,03541d8d436a25eb9f0da8582930647a58168a6c,gabinsane_multimodal-vae-comparison
2,03a1e7075edfe00bea813019c58c9861bdc305ab,gabinsane_multimodal-vae-comparison
3,03d8331004cbdae249a2fea07837ca8e1dab88e4,gabinsane_multimodal-vae-comparison
4,04977864f89f12e320905b26a3732515ac675982,gabinsane_multimodal-vae-comparison


In [47]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

 10%|███▎                               | 180/1879 [03:04<28:58,  1.02s/it]

Got Errors {'0ac2919c2e13d0e389f878e8feed5f9f714b6ca3': 'Key 0ac2919c2e13d0e389f878e8feed5f9f714b6ca3 not found in /da5_fast/All.sha1c/commit_10.tch'}


 10%|███▍                               | 183/1879 [03:07<28:56,  1.02s/it]

Got Errors {'0e81f04db03fd179f481c86ddb654294951e5c13': 'Key 0e81f04db03fd179f481c86ddb654294951e5c13 not found in /da5_fast/All.sha1c/commit_14.tch'}


 10%|███▍                               | 186/1879 [03:10<28:50,  1.02s/it]

...

Got Errors {'24b83209947cba207ea50c28e863dff6895eece9': 'Key 24b83209947cba207ea50c28e863dff6895eece9 not found in /da5_fast/All.sha1c/commit_36.tch'}


 99%|█████████████████████████████████▌| 1858/1879 [31:40<00:21,  1.03s/it]

Got Errors {'7e3021081c9402eadf92bb5dd3fb70ef079108dd': 'Key 7e3021081c9402eadf92bb5dd3fb70ef079108dd not found in /da5_fast/All.sha1c/commit_126.tch'}


100%|██████████████████████████████████| 1879/1879 [32:01<00:00,  1.02s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,034f34e1158ffb69f5ad4ceefeff4991e312abc8,608d6eb0d47316c88c6fdb43ce04e0ae63b59817,[4057cf61289ab2913704884d963d0d7647e6d45c],Gabriela Šejnová <g.sejnova@gmail.com>,1661779711,+0200,Gabriela Šejnová <g.sejnova@gmail.com>,1661779711,+0200,test_torchmmvae.py passe\n,034f34e1158ffb69f5ad4ceefeff4991e312abc8,gabinsane_multimodal-vae-comparison
1,03541d8d436a25eb9f0da8582930647a58168a6c,e26b83f873b7a7cbde2fe1518d679aa8a9066e0c,[62ed61628a8a787c5262ea225a7bb1ee1aaf03e0],Gabriela Šejnová <g.sejnova@gmail.com>,1664540067,+0200,Gabriela Šejnová <g.sejnova@gmail.com>,1664540067,+0200,mnist_svhn vis and labelled tsne\n,03541d8d436a25eb9f0da8582930647a58168a6c,gabinsane_multimodal-vae-comparison


In [53]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '034f34e1158ffb69f5ad4ceefeff4991e312abc8', 'tree': '608d6eb0d47316c88c6fdb43ce04e0ae63b59817', 'parent': ['4057cf61289ab2913704884d963d0d7647e6d45c'], 'author': 'Gabriela Šejnová <g.sejnova@gmail.com>', 'author_time': 1661779711, 'author_tz': '+0200', 'committer': 'Gabriela Šejnová <g.sejnova@gmail.com>', 'committer_time': 1661779711, 'committer_tz': '+0200', 'message': 'test_torchmmvae.py passe\n'}


In [54]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)

In [57]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,geoknow_linkedgeodata,034f34e1158ffb69f5ad4ceefeff4991e312abc8,Gabriela Šejnová <g.sejnova@gmail.com>,1661779711,test_torchmmvae.py passe\n


In [69]:
#mode a means append, so you have all your projects in the same file
yournetid='sbombry1'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=',', mode='a', header=False)

In [37]:
summary = df_commit_data.groupby('project').agg(
    num_commits=('commit', 'nunique'),
    num_authors=('author', 'nunique'),
    max_time=('author_time', 'max'),
    min_time=('author_time', 'min'),
)
summary

,num_commits,num_authors,max_time,min_time
project,,,,
arfc_pride,995,14,1740328556,1576622284
fairrootgroup_fairroot,7215,99,1774830205,1216626209
gabinsane_multimodal-vae-comparison,432,9,1747690487,1624959492
geoknow_linkedgeodata,431,18,1732125354,1359478118
geospace-code_pymap3d,1339,49,1751955558,1407040083
mansmeg_posteriordb,1290,40,1758720825,1554791725
markmfredrickson_optmatch,1967,29,1772727653,1307046210
nexpy_nexpy,3453,22,1774644547,1369843716
statistikat_vim,955,31,1775046232,1378979956


In [38]:
# Mnaual github review (stars, forks, last_commit)
# Project          | Num_stars | Num_forks | Last_commit_Date 
# arfc_pride             3           7        Feb 11th 2025
# Fairrootgroup          70          100      Aug 6th 2025
# Multimodal-vae         34          11       May 19th 2025
# linkedgeodata          162         33       Sep 29th 2024
# pymap3d                445         88       Jun 21st 2026
# posteriordb            232         52       Sep 25th 2026
# optmatch               50          14       Sep 24th 2026
# nexpy                  53          25       Sep 21st 2026
# vim                    93          15       Sep 22nd 2026
# tableone               197         52       Feb 10th 2026